# 2.1 Intent Classifier — CLO04 (Classification)

**Project M.O.N.D.A.Y** · Course 24CSE0316 · Labs 14–20 (Logistic Regression, SVM, KNN, Decision Tree).

**Goal.** Route a raw user command into one of 6 intents so high-confidence commands skip the (slow, expensive) LLM Planner. We build the pipeline `text → clean → TF-IDF → classifier`, **formally compare five models**, and deploy the winner.

**CLO04 evidence produced here:** confusion matrix, accuracy / precision / recall / F1 (macro + per class), and a ranked model comparison.

> Run `python -m datagen.generate` first to create `data/synthetic/commands.csv`.

In [ ]:
import sys, os
# allow imports of the project packages when running from notebooks/
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from common.preprocessing import build_vectorizer
from common.metrics import classification_metrics
from datagen.config import INTENTS, RANDOM_SEED

## 1. Load data & inspect class balance

The dataset is synthesized (`datagen/commands.py`): templated commands per intent + shared filler vocabulary + ~6% injected label noise to simulate real annotation ambiguity. That noise is deliberate — it keeps the task learnable-but-not-trivial so the 5-model comparison is meaningful.

In [ ]:
df = pd.read_csv('../data/synthetic/commands.csv')
print('rows:', len(df))
display(df.sample(8, random_state=0))
df['intent'].value_counts().plot.bar(title='Command count per intent', rot=45);

## 2. Preprocessing pipeline (feature engineering — CLO03)

`text → clean_text → TF-IDF (unigrams + bigrams)`. Each model is wrapped in its own `Pipeline` so the comparison is fully independent.

In [ ]:
X, y = df['text'].values, df['intent'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print(f'train={len(X_train)}  test={len(X_test)}')

def make_pipeline(model):
    return Pipeline([('tfidf', build_vectorizer()), ('clf', model)])

## 3. Compare five models

Logistic Regression · KNN · Decision Tree · Random Forest · SVM.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, C=4.0),
    'KNN': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1),
    'SVM': SVC(kernel='linear', C=1.0, probability=True, random_state=RANDOM_SEED),
}

rows, preds = [], {}
for name, model in models.items():
    pipe = make_pipeline(model).fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    preds[name] = y_pred
    m = classification_metrics(y_test, y_pred, labels=INTENTS)
    rows.append({'model': name, 'accuracy': m['accuracy'],
                 'precision_macro': m['precision_macro'],
                 'recall_macro': m['recall_macro'], 'f1_macro': m['f1_macro']})

comparison = pd.DataFrame(rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
comparison.round(4)

In [ ]:
ax = comparison.set_index('model')['f1_macro'].plot.barh(
    title='Macro-F1 by model', xlim=(0.75, 1.0))
ax.invert_yaxis();

## 4. Winner: confusion matrix + per-class report

In [ ]:
best = comparison.iloc[0]['model']
print('Best model:', best)
print(classification_report(y_test, preds[best], zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, preds[best], labels=INTENTS, xticks_rotation=45,
    cmap='Blues', ax=ax, colorbar=False)
ax.set_title(f'Confusion Matrix ({best})');

## 5. Conclusions

- **Linear models win.** Logistic Regression and linear SVM lead; both handle high-dimensional sparse TF-IDF well. KNN is close behind.
- **Decision Tree trails clearly** — it overfits the injected label noise, which is exactly why a single tree underperforms the ensemble (Random Forest) and the linear models. This spread is what makes the comparison worth doing.
- **Accuracy tops out near the noise ceiling (~94%)** because ~6% of labels are intentionally ambiguous; a model scoring 100% here would signal data leakage, not skill.
- **Deployment:** the winner is refit on the full dataset and saved to `ml/intent_classifier/model.joblib` by `train.py`. Production inference (`ml/intent_classifier/classifier.py`) returns `(intent, confidence)`; if confidence < threshold the command is routed to `llm_fallback` → the Planner, which feeds the RL router (2.5).

Reproduce the saved artifacts with: `python -m ml.intent_classifier.train`.